In [ ]:
from transformers import BertTokenizer, BertModel
import torch.nn as nn
import torch
import numpy as np
from tqdm import tqdm
from datasets import load_dataset

In [ ]:
# Load the AG News dataset
dataset = load_dataset('fancyzhx/ag_news')
# Split the dataset into training and test sets
train_dataset = dataset['train']
test_dataset = dataset['test']
# Explore the dataset
print(f"Number of training samples: {len(train_dataset)}")
print(f"Number of test samples: {len(test_dataset)}")
print(train_dataset[0])
label_names = ['World', 'Sports', 'Business', 'Sci/Tech']
# Extract texts and labels
train_texts = list(train_dataset['text'])
train_labels = list(train_dataset['label'])
train_labels = torch.tensor(train_labels)
test_texts = list(test_dataset['text'])
test_labels = list(test_dataset['label'])
test_labels = torch.tensor(test_labels)
# Initialize the tokenizer


In [ ]:
# Initialize the tokenizer
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
# Tokenize the dataset
train_encodings = tokenizer(train_texts, truncation=True,
padding=True, max_length=128,return_tensors='pt')
test_encodings = tokenizer(test_texts, truncation=True,
padding=True, max_length=128,return_tensors='pt')
# Load the pre-trained BERT model
model = BertModel.from_pretrained('bert-base-uncased')
# Move model to GPU if available
device = torch.device('cuda' if torch.cuda.is_available() else'cpu')
model.to(device)

In [ ]:
# Freeze all the layers of the BERT model so they won't be trained
for param in model.parameters():
    param.requires_grad = False
# Define a classification layer with 4 output nodes
# 4 classes (World, Sports, Business, Sci/Tech)
classifier = nn.Linear(model.config.hidden_size, 4)
classifier.to(device)

In [ ]:
from torch.utils.data import DataLoader, TensorDataset
from torch.optim import AdamW
# Create TensorDataset for DataLoader
train_dataset = TensorDataset(train_encodings['input_ids'],
train_encodings['attention_mask'], train_labels)
test_dataset = TensorDataset(test_encodings['input_ids'],
test_encodings['attention_mask'], test_labels)
# Create DataLoader
train_loader = DataLoader(train_dataset, batch_size=50, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=50, shuffle=False)

In [ ]:
# Define the optimizer for the classifier layer
optimizer = AdamW(classifier.parameters(), lr=1e-4)
# Define loss function
loss_fn = nn.CrossEntropyLoss()

In [ ]:
# Training loop
def train_model(model, classifier, train_loader, optimizer,
                loss_fn, device, epochs=10):
    model.eval() # We don't want to train the full BERT model, so
    # set it to eval mode
    classifier.train() # We want to train the classifier layer
    # (top layer (dim=-1))
    for epoch in range(epochs):
        # print(f"Epoch {epoch+1}/{epochs}")
        epoch_loss = 0
        #batch is a tuple containing a batch of inputs (tokenized
        # text), attention
        # masks, and labels.
        for batch in tqdm(train_loader):
            input_ids = batch[0].to(device)
            attention_mask = batch[1].to(device)
            labels = batch[2].to(device)
            # Forward pass through BERT (we don't update these weights)
            with torch.no_grad():
                outputs = model(input_ids=input_ids,attention_mask=attention_mask)
                cls_embeddings = outputs.last_hidden_state[:, 0, :]
                # [CLS] token representation
            # Forward pass through the classifier
            logits = classifier(cls_embeddings)
            # Calculate loss
            loss = loss_fn(logits, labels)
            # Backward pass and optimization
            optimizer.zero_grad()
            #Before we compute the gradients for the current batch, we
            # need to reset
            loss.backward()
            #This line performs the optimization step,
            #where the actual model parameters (weights) are updated
            #based on the gradients calculated in the previous step.
            optimizer.step()
            epoch_loss += loss.item()
        #print(f"Training loss: {epoch_loss/len(train_loader)}")
            
# Train the model
train_model(model, classifier, train_loader, optimizer, loss_fn,
device, epochs=10)

In [ ]:
def extract_features(texts):
    features = []
    batch_size = 16
    for i in tqdm(range(0, len(texts), batch_size)):
        batch_texts = texts[i:i+batch_size]
        # Tokenize the batch
        encoding = tokenizer(batch_texts, padding=True,
                             truncation=True,max_length=128, return_tensors='pt')
        # Move tensors to device
        input_ids = encoding['input_ids'].to(device)
        attention_mask = encoding['attention_mask'].to(device)
        with torch.no_grad():
            outputs = model(input_ids, attention_mask=attention_mask)
        # Use the [CLS] token representation as the feature
        cls_embeddings = outputs.last_hidden_state[:, 0, :]
        # Shape: (for all batch,0th token <CLS>,entire embedding length)
        # Move embeddings to CPU and convert to numpy array
        features.append(cls_embeddings.cpu().numpy())
    features = np.concatenate(features, axis=0)
    return features

In [12]:
texts = ["这个东西很好用", "质量很差不推荐"]
vecs = extract_features(texts)
y=classifier(torch.tensor(vecs).to(device))
y

100%|██████████| 1/1 [00:00<00:00,  3.59it/s]


tensor([[ 1.2340, -1.2525, -1.1574,  0.4092],
        [ 1.3049, -1.3519, -0.8895,  0.2944]], device='cuda:0',
       grad_fn=<AddmmBackward0>)